# Flux LoRA Train

⚠️ **Remember to copy this notebook in your Drive and rename.**


**This workflow uses AI Toolkit'**

[jhj0517/finetuning-notebooks](https://github.com/jhj0517/finetuning-notebooks)

[ostris/ai-toolkit](https://github.com/ostris/ai-toolkit)

**HuggingFace Alternative**

[README_flux.md](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/README_flux.md)

[train_dreambooth_lora_flux.py](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/train_dreambooth_lora_flux.py)

[test_dreambooth_lora_flux.py](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/test_dreambooth_lora_flux.py)

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

##Confirm using A100 GPU

You absolutely need an A100 to train a Flux LoRA. Sometimes Colab says that it is connected to an A100 while secretly connecting to a lower GPU. Confirm below that you actually connected to an A100 as this isn't going to work if you are connected to a lesser GPU.

In [ ]:
!nvidia-smi

Tue May 26 23:45:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   53C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Install

In [ ]:
!git clone https://github.com/ostris/ai-toolkit.git
%cd ai-toolkit
!git submodule update --init --recursive > /dev/null 2>&1
!pip install --quiet -r requirements.txt
!pip install -q "transformers>=4.52.0" "diffusers>=0.33.0"

# Downgrade Numpy
!pip uninstall -y numpy
!pip install --quiet --force-reinstall --no-deps numpy==1.26.3

fatal: destination path 'ai-toolkit' already exists and is not an empty directory.
/content/ai-toolkit
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Found existing installation: numpy 1.26.3
Uninstalling numpy-1.26.3:
  Successfully uninstalled numpy-1.26.3


In [ ]:
# Restart session
import os
os.kill(os.getpid(), 9)
print("IGNORE WARNING. This was done intentionally.")
# Continue to the next cell and keep going after runtime restarts

## Setup

In [ ]:
# Fix numpy incompatibility from https://github.com/ostris/ai-toolkit/issues/267
import numpy
print("Numpy should be 1.26.3. The version currently being used is: " + numpy.__version__)

Numpy should be 1.26.3. The version currently being used is: 1.26.3


## Mount Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Hugging Face Token

In [ ]:
# Sign up at Hugging Face and create a "Read" access token (not the default "Fine-Grained" token).
# Click the 🔑 "Secrets" icon in the left sidebar.
# Enable Notebook Access, Set the Name to "HF_TOKEN", Paste your token as the Value

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

## Set Directories

In [ ]:
import os

DATASET_DIR = '/content/drive/MyDrive/GenAi_RaniaRamón/FineTuning/dataset_ranram_diagrams/style01'
OUTPUT_DIR  = '/content/drive/MyDrive/GenAi_RaniaRamón/FineTuning/weights_ranram_diagrams'
LORA_NAME   = 'ranram_arch_diagram'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Login to Weights & Biases with token

In [ ]:
# Get your API token from https://wandb.ai/settings
# Click the empty space after "quit:" to enter code, or click 2, then API key
!pip install wandb -q
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: gramonga4434 (gramonga4434-iaac) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Config

In [ ]:
!ls /content/ai-toolkit/extensions_built_in/diffusion_models/

chroma	     flux2	   __init__.py	  omnigen2    zeta_chroma
ernie_image  flux_kontext  ltx2		  qwen_image  z_image
f_light      hidream	   nucleus_image  wan22


In [ ]:
import os
import sys
sys.path.append('/content/ai-toolkit')
from toolkit.job import run_job
from collections import OrderedDict
from PIL import Image
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
# Model
repo_id_or_path = 'black-forest-labs/FLUX.2-klein-base-4B'
quantize = True
quantize_te = True
low_vram = True
dtype = "bf16"

# Training Steps (max_steps)
# More steps = more learning, higher risk of overfitting
# Fewer steps = less learning, may underfit
# 1000  = quick test / smoke run
# 1500  = good for small datasets with low LR
# 2000  = default starting point (30 images)
# 3000  = larger datasets or lower LR runs
# 4000  = high variety datasets, risk of overfit on small sets
steps = 2000

# Saving checkpoints
# Saves a checkpoint every save_every steps
# Deletes oldest checkpoint once max_step_saves_to_keep is exceeded
save_every = 250
max_step_saves_to_keep = 10

# Sampling
# Reduced resolution and frequency to avoid VRAM spikes mid-training
sample_every = 500
sample_seed = 77
sample_steps = 30
width = 768
height = 768

# Learning Rate
# Higher = faster learning, more risk of overfitting
# Lower  = slower learning, more stable convergence
# 4e-4 = 0.0004  (FLUX.1 default, aggressive)
# 2e-4 = 0.0002  (moderate)
# 1e-4 = 0.0001  (conservative, recommended for FLUX.2)
# 5e-5 = 0.00005 (slow, stable, good for small datasets)
# 1e-5 = 0.00001 (very slow, use if overfitting persists)
lr = 1e-4

# Replace Prompts
sample_prompt_1 = "ranram_arch_diagram stepped terraces, green roofs, courtyard void, mid-rise housing, pixelated massing, green sedum, glass curtain wall, white framing, 3D model, simplified massing, boxy white volumes, low-poly massing, thick outer profiles, Bold outline, thin interior linework, acid yellow-green, pure white, red accents, ambient occlusion, soft subtle shading"
sample_prompt_2 = "ranram_arch_diagram oval ring massing, stepped pixelated terraces, central courtyard, mid-rise housing, sloped roofscape, matte white surface, green roof vegetation, 3D model, simplified massing, boxy white volumes, low-poly massing, thick outer profiles, Bold outline, thin interior linework, pure white, lime green, light gray, ambient occlusion, soft subtle shading."
sample_prompt_3 = "ranram_arch_diagram memorial park, iconic skyscraper cluster with central sunken voids, cluster defined by the negative space , white matte resin, digital wireframe, glass curtain wall textures, 3D model, simplified massing, boxy white volumes, low-poly massing, thick outer profiles, Bold outline, thin interior linework, bright lime green, sky blue, stark white, charcoal grey, ghosted grey, ambient occlusion, soft subtle shading."
sample_prompts = [sample_prompt_1, sample_prompt_2, sample_prompt_3]

# Advanced
# Lora Network
linear = 16
linear_alpha = 16

# Dataset
caption_ext = "txt"
caption_dropout_rate = 0.05
shuffle_tokens = False
cache_latents_to_disk = True
resolution = [1024]

# Training
performance_log_every = 1000
train_only_specific_layers = False
only_if_contains = ["transformer.single_transformer_blocks.7.proj_out", "transformer.single_transformer_blocks.20.proj_out"]
batch_size = 1
gradient_accumulation_steps = 1
train_dtype = "bf16"
train_unet = True
train_text_encoder = False   # Qwen3 stays frozen, big VRAM saving
content_or_style = 'balanced'
gradient_checkpointing = True
noise_scheduler = 'flowmatch'
optimizer = 'adamw8bit'
use_ema = False              # EMA duplicates trainable weights, disable on 16 GB
ema_decay = 0.99             # ignored when use_ema = False

## Train

In [ ]:
job_to_run = OrderedDict([
    ('job', 'extension'),
    ('config', OrderedDict([
        ('name', LORA_NAME),
        ('process', [
            OrderedDict([
                ('type', 'sd_trainer'),
                ('training_folder', OUTPUT_DIR),
                ('performance_log_every', 1000),
                ('device', 'cuda:0'),
                ('network', OrderedDict([
                    ('type', 'lora'),
                    ('linear', linear),
                    ('linear_alpha', linear_alpha),
                    ('network_kwargs', OrderedDict([
                      ("prefix", [
                          'model',
                          'transformer.single_transformer_blocks',
                          'transformer.transformer_blocks'
                      ])
                    ]))
                ])),
                ('save', OrderedDict([
                    ('dtype', dtype),
                    ('save_every', save_every),
                    ('max_step_saves_to_keep', max_step_saves_to_keep)
                ])),
                ('datasets', [
                    OrderedDict([
                        ('folder_path', DATASET_DIR),
                        ('caption_ext', caption_ext),
                        ('caption_dropout_rate', caption_dropout_rate),
                        ('shuffle_tokens', shuffle_tokens),
                        ('cache_latents_to_disk', cache_latents_to_disk),
                        ('resolution', resolution)
                    ])
                ]),
                ('train', OrderedDict([
                    ('batch_size', batch_size),
                    ('steps', steps),
                    ('gradient_accumulation_steps', gradient_accumulation_steps),
                    ('train_unet', train_unet),
                    ('train_text_encoder', train_text_encoder),
                    ('content_or_style', content_or_style),
                    ('gradient_checkpointing', gradient_checkpointing),
                    ('noise_scheduler', noise_scheduler),
                    ('optimizer', optimizer),
                    ('lr', lr),
                    ('ema_config', OrderedDict([
                        ('use_ema', use_ema),
                        ('ema_decay', ema_decay)
                    ])),
                    ('dtype', train_dtype)
                ])),
                ('model', OrderedDict([
                    ('name_or_path', repo_id_or_path),
                    ('arch', 'flux2_klein_4b'),
                    ('quantize', quantize),
                ])),
                ('sample', OrderedDict([
                    ('sampler', 'flowmatch'),
                    ('sample_every', sample_every),
                    ('width', width),
                    ('height', height),
                    ('prompts', sample_prompts),
                    ('neg', ''),
                    ('seed', sample_seed),
                    ('walk_seed', True),
                    ('guidance_scale', 4),
                    ('sample_steps', sample_steps)
                ]))
            ])
        ])
    ])),
    ('meta', OrderedDict([
        ('name', '[name]'),
        ('version', '1.0')
    ]))
])

# Conditional Parameters
if train_only_specific_layers:
    network = job_to_run["config"]["process"][0]["network"]
    network_kwargs = network.setdefault("network_kwargs", OrderedDict())
    network_kwargs["only_if_contains"] = only_if_contains


wandb.init(project='flux2-lora', name=LORA_NAME, resume='allow')

run_job(job_to_run)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.15). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


{
    "type": "sd_trainer",
    "training_folder": "/content/drive/MyDrive/GenAi_RaniaRamo\u0301n/FineTuning/weights_FLUX.2",
    "performance_log_every": 1000,
    "device": "cuda:0",
    "network": {
        "type": "lora",
        "linear": 16,
        "linear_alpha": 16,
        "network_kwargs": {
            "prefix": [
                "model",
                "transformer.single_transformer_blocks",
                "transformer.transformer_blocks"
            ]
        }
    },
    "save": {
        "dtype": "bf16",
        "save_every": 250,
        "max_step_saves_to_keep": 10
    },
    "datasets": [
        {
            "folder_path": "/content/drive/MyDrive/GenAi_RaniaRamo\u0301n/FineTuning/dataset_FLUX.2",
            "caption_ext": "txt",
            "caption_dropout_rate": 0.05,
            "shuffle_tokens": false,
            "cache_latents_to_disk": true,
            "resolution": [
                1024
            ]
        }
    ],
    "train": {
        "batch_size

flux-2-klein-base-4b.safetensors:   0%|          | 0.00/7.75G [00:00<?, ?B/s]

Keeping transformer on CPU for quantization
Quantizing Transformer
 - quantizing 25 transformer blocks


100%|██████████| 25/25 [00:04<00:00,  5.34it/s]


...


Saving at step 1500
Saved checkpoint to /content/drive/MyDrive/GenAi_RaniaRamón/FineTuning/weights_FLUX.2/big_ran_ram/big_ran_ram_000001500.safetensors
Saved optimizer to /content/drive/MyDrive/GenAi_RaniaRamón/FineTuning/weights_FLUX.2/big_ran_ram/optimizer.pt



big_ran_ram:  87%|████████▋ | 1749/2000 [1:37:49<13:58,  3.34s/it, lr: 1.0e-04 loss: 4.380e-01]


Saving at step 1750
Saved checkpoint to /content/drive/MyDrive/GenAi_RaniaRamón/FineTuning/weights_FLUX.2/big_ran_ram/big_ran_ram_000001750.safetensors
Saved optimizer to /content/drive/MyDrive/GenAi_RaniaRamón/FineTuning/weights_FLUX.2/big_ran_ram/optimizer.pt


big_ran_ram: 100%|█████████▉| 1999/2000 [1:51:44<00:03,  3.35s/it, lr: 1.0e-04 loss: 8.050e-01]



Saved checkpoint to /content/drive/MyDrive/GenAi_RaniaRamón/FineTuning/weights_FLUX.2/big_ran_ram/big_ran_ram.safetensors
Saved optimizer to /content/drive/MyDrive/GenAi_RaniaRamón/FineTuning/weights_FLUX.2/big_ran_ram/optimizer.pt


## Disconnect

In [ ]:
from google.colab import runtime
runtime.unassign()